# GROUP 4 - Air Pollution Forecasting and Anomalous Sensor Readings

**Research question.** How sensitive are predictions of an air pollutant to (a) the way we validate over time, (b) the way we handle missing values, and (c) the removal of anomalous sensor readings from the training data?

**Team:** Sai Rohith CHETLAPALLY   Srija POUDAPALLY|  **Dataset:** UCI Air Quality (De Vito et al., 2008)  |  **Target:** `CO(GT)`

**How to run.** Colab: *Runtime > Run all*. The notebook looks for `AirQualityUCI.csv` next to it, then tries to download it from UCI, and finally asks you to upload it. Everything is seeded (`SEED`), so a second run gives the same numbers.




## Step 0 - Setup, seeds and experiment log

All knobs you may be asked to change live in the **config cell** below (split ratio, contamination values, imputer, seeds). In the oral defense, change one of them and re-run from Step 2.

In [ ]:
import os, io, zipfile, urllib.request, warnings, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor, IsolationForest
from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, median_absolute_error
from sklearn.decomposition import PCA

warnings.filterwarnings("ignore")

# ------------------------------- CONFIG (change here) -------------------------------
SEED = 42                          # main seed, used by every random component
SEEDS = [42, 7, 123]               # extra seeds to measure run-to-run noise in the experiments
TARGET = "CO(GT)"                  # prediction target (mg/m3)
PROXY_COLS = ["PT08.S1(CO)"]       # sensor that is a direct proxy of the target -> excluded
TRAIN_FRAC = 0.75                  # chronological split: first 75% of time = train, last 25% = test
CONTAMINATIONS = [0.01, 0.03, 0.05]  # Isolation Forest contamination values for Experiment 3
IF_CONTAM_MAIN = 0.03              # contamination used for the Step 4 characterisation
FFILL_LIMIT = 6                    # max consecutive hours carried forward in the ffill imputer
# ------------------------------------------------------------------------------------

random.seed(SEED)
np.random.seed(SEED)
os.makedirs("figures", exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3, "axes.spines.top": False, "axes.spines.right": False})

# ---- experiment log: one row per (experiment, setting, model) ----
LOG = []
def log(experiment, setting, model, mae, rmse, notes=""):
    LOG.append(dict(experiment=experiment, setting=setting, model=model,
                    MAE=round(float(mae), 4), RMSE=round(float(rmse), 4), notes=notes))

def metrics(y_true, y_pred):
    return dict(MAE=mean_absolute_error(y_true, y_pred),
                RMSE=float(np.sqrt(mean_squared_error(y_true, y_pred))),
                R2=r2_score(y_true, y_pred),
                MedAE=median_absolute_error(y_true, y_pred))

print("Setup done. SEED =", SEED)

## Step 1 - Inspect the data, check leakage, state two hypotheses

The UCI file uses `;` as separator, `,` as decimal mark, and **-200 as the code for a missing value**. It also has two empty trailing columns and some empty trailing rows.

In [ ]:
def load_air_quality():
    """Load the raw AirQualityUCI.csv: local file -> UCI download -> Colab upload."""
    for p in ["AirQualityUCI.csv", "/content/AirQualityUCI.csv", "/content/drive/MyDrive/AirQualityUCI.csv"]:
        if os.path.exists(p):
            print("Loaded local file:", p)
            return pd.read_csv(p, sep=";", decimal=",")
    try:
        url = "https://archive.ics.uci.edu/static/public/360/air+quality.zip"
        data = urllib.request.urlopen(url, timeout=30).read()
        z = zipfile.ZipFile(io.BytesIO(data))
        name = [n for n in z.namelist() if n.endswith("AirQualityUCI.csv")][0]
        print("Downloaded from UCI:", url)
        return pd.read_csv(z.open(name), sep=";", decimal=",")
    except Exception as e:
        print("Download failed (", type(e).__name__, "). Falling back to manual upload.")
    try:
        from google.colab import files
        up = files.upload()
        return pd.read_csv(list(up)[0], sep=";", decimal=",")
    except ImportError:
        raise FileNotFoundError("Put AirQualityUCI.csv next to this notebook (download it from "
                                "https://archive.ics.uci.edu/dataset/360/air+quality).")

raw = load_air_quality()
raw = raw.dropna(axis=1, how="all").dropna(axis=0, how="all")        # drop empty trailing columns / rows
raw["datetime"] = pd.to_datetime(raw["Date"].astype(str) + " " + raw["Time"].astype(str),
                                 format="%d/%m/%Y %H.%M.%S")
df = (raw.drop(columns=["Date", "Time"]).set_index("datetime").sort_index())
df = df[~df.index.duplicated()]
df = df.apply(pd.to_numeric, errors="coerce").replace(-200, np.nan)  # -200 means "missing"
print("Shape:", df.shape, "| period:", df.index.min(), "->", df.index.max())
df.head()

In [ ]:
miss = df.isna().mean().sort_values(ascending=False).rename("missing_fraction").to_frame()
miss["missing_count"] = df.isna().sum()
display(miss.round(3))
print("\nNMHC(GT) is almost entirely missing -> unusable as a target.")
print(f"Target {TARGET}: {df[TARGET].notna().sum()} usable hours out of {len(df)}")

In [ ]:
y_series = df[TARGET]
fig, ax = plt.subplots(1, 3, figsize=(16, 3.8))
ax[0].hist(y_series.dropna(), bins=50, color="#4C78A8")
ax[0].set(title=f"Fig 1a - Distribution of {TARGET}", xlabel="mg/m3", ylabel="hours")
ax[1].plot(y_series.index, y_series, lw=0.4, color="#888888", label="hourly")
ax[1].plot(y_series.index, y_series.rolling(24 * 7, min_periods=24).mean(), lw=1.5, color="#E45756", label="7-day mean")
ax[1].set(title=f"Fig 1b - {TARGET} over time", ylabel="mg/m3"); ax[1].legend()
prof = y_series.groupby(y_series.index.hour).mean()
ax[2].plot(prof.index, prof.values, marker="o", color="#54A24B")
ax[2].set(title="Fig 1c - Mean by hour of day", xlabel="hour", ylabel="mg/m3")
plt.tight_layout(); plt.savefig("figures/fig1_target_overview.png", bbox_inches="tight"); plt.show()

print(f"Lag-1 autocorrelation of the hourly target: {y_series.autocorr(1):.3f}")
print(f"Lag-24 autocorrelation:                     {y_series.autocorr(24):.3f}")

### Leakage check and target justification

We correlate every column with the target. Columns that are essentially *another measurement of the same quantity* must not be used as features:

* `NMHC(GT)`, `C6H6(GT)`, `NOx(GT)`, `NO2(GT)` are other **reference-analyzer outputs** - they are ground truth of neighbouring pollutants and are not available from the cheap sensor array. They are excluded.
* `PT08.S1(CO)` is the tin-oxide sensor **built to measure CO**, i.e. a direct proxy of the target. It is excluded (an ablation below quantifies how much it would help).

**Why `CO(GT)`?** It has a moderate share of missing values (so Experiment 2 is meaningful), a clear daily cycle, and it is not mostly missing like `NMHC(GT)`.

In [ ]:
corr = df.corr(numeric_only=True)[TARGET].drop(TARGET)
corr = corr.reindex(corr.abs().sort_values(ascending=False).index).rename("corr_with_target").to_frame()
corr["decision"] = ["EXCLUDED (reference analyzer)" if c.endswith("(GT)") else
                    ("EXCLUDED (direct proxy)" if c in PROXY_COLS else "kept as feature") for c in corr.index]
display(corr.round(3))

### Hypotheses (written BEFORE any model is trained)

* **H1 - validation protocol.** A random split will give a *lower* test error than a chronological split, because neighbouring hours are strongly autocorrelated (a random test hour almost always has a training hour right next to it) and because the pollutant level drifts with the season.
* **H2 - anomaly removal.** Removing the points flagged by Isolation Forest from the *training* set will **not** noticeably reduce the test error, because many statistically unusual readings are real pollution episodes (rush hour, cold nights) rather than sensor faults, so deleting them removes useful information.

Both hypotheses are checked automatically in Step 6.

## Step 2 - Fixed evaluation protocol and reproducible pipeline

**Protocol (fixed for the whole project).**

* Rows without a target value are dropped (we never impute the target).
* **Chronological hold-out:** first `TRAIN_FRAC` of the remaining rows (in time order) = train, the rest = test. No shuffling.
* Every preprocessing step (imputer, scaler, anomaly detector) is inside a scikit-learn `Pipeline` or fitted on **train only**; test data is never used for imputation statistics, tuning or anomaly removal.
* Features: 4 remaining sensors (`PT08.S2` to `PT08.S5`), `T`, `RH`, `AH`, plus calendar features `hour`, `dayofweek`, `month`. No lag features of the target.
* Metrics: **MAE**, **RMSE**, R2 and median absolute error.

In [ ]:
GT_OTHERS = [c for c in df.columns if c.endswith("(GT)") and c != TARGET]
feat = df.drop(columns=GT_OTHERS + [TARGET] + PROXY_COLS)
SENSOR_COLS = list(feat.columns)                       # physical measurements (with NaN)
X_all = feat.copy()
X_all["hour"] = X_all.index.hour
X_all["dayofweek"] = X_all.index.dayofweek
X_all["month"] = X_all.index.month
CAL_COLS = ["hour", "dayofweek", "month"]
y_all = df[TARGET]

valid = y_all.notna()
X_v, y_v = X_all[valid], y_all[valid]
n_train = int(len(X_v) * TRAIN_FRAC)
X_train, X_test = X_v.iloc[:n_train], X_v.iloc[n_train:]
y_train, y_test = y_v.iloc[:n_train], y_v.iloc[n_train:]

print("Features:", list(X_all.columns))
print(f"Train: {len(X_train)} rows  {X_train.index.min().date()} -> {X_train.index.max().date()}")
print(f"Test : {len(X_test)} rows  {X_test.index.min().date()} -> {X_test.index.max().date()}")
print(f"Mean target  train={y_train.mean():.3f}  test={y_test.mean():.3f}   (seasonal drift between the two blocks)")
print("Test months never seen in training:", sorted(set(X_test.month) - set(X_train.month)))

In [ ]:
def make_model(name, seed):
    if name == "RF":
        return RandomForestRegressor(n_estimators=200, min_samples_leaf=3, n_jobs=-1, random_state=seed)
    if name == "HGB":
        return HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05, max_leaf_nodes=31, random_state=seed)
    raise ValueError(name)

def make_pipeline(name, imputer="median", seed=SEED):
    """imputer: 'median' | 'knn' | 'none' (HGB only: uses its native NaN support).
    The imputer sits INSIDE the pipeline, so it is fitted on training data only."""
    steps = []
    if imputer == "median":
        steps.append(("imputer", SimpleImputer(strategy="median")))
    elif imputer == "knn":
        steps += [("scaler", StandardScaler()), ("imputer", KNNImputer(n_neighbors=5))]
    elif imputer == "none":
        assert name == "HGB", "only HistGradientBoosting accepts NaN natively"
    steps.append(("model", make_model(name, seed)))
    return Pipeline(steps)

def make_iso_prep():
    """Pre-processing for the anomaly detector: median-impute + standardise (fit on train only)."""
    return Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())])

print(make_pipeline("RF"))

In [ ]:
# ---- Ablation for the leakage decision: how much would the proxy sensor help? (not used afterwards) ----
Xp = X_all.copy(); Xp["PT08.S1(CO)"] = df["PT08.S1(CO)"]
Xp_v = Xp[valid]
m_clean = make_pipeline("RF").fit(X_train, y_train)
m_leaky = make_pipeline("RF").fit(Xp_v.iloc[:n_train], y_train)
r_clean = metrics(y_test, m_clean.predict(X_test)); r_leaky = metrics(y_test, m_leaky.predict(Xp_v.iloc[n_train:]))
print(f"RF without proxy sensor : RMSE={r_clean['RMSE']:.3f}  R2={r_clean['R2']:.3f}")
print(f"RF WITH   proxy sensor  : RMSE={r_leaky['RMSE']:.3f}  R2={r_leaky['R2']:.3f}")
log("Leakage ablation", "with PT08.S1(CO) (not used)", "RF", r_leaky["MAE"], r_leaky["RMSE"], "shows why the proxy is excluded")

## Step 3 - Supervised models

Two tree ensembles are trained with the chronological protocol and compared with two naive baselines:

* **Mean baseline** - always predict the training mean.
* **Persistence baseline** - predict the previous hour's CO (a strong baseline for an autocorrelated series; falls back to the training mean when the previous hour is missing).

**Metrics beyond "accuracy".** For regression accuracy is undefined, so we report
* **MAE** - robust, in the same units as CO (mg/m3), easy to interpret;
* **RMSE** - squares errors, so it punishes the large misses that matter most for pollution peaks;
* a large **RMSE/MAE ratio** therefore signals that a model is mostly good but occasionally badly wrong; R2 tells how much variance is explained compared with the mean baseline.

In [ ]:
res3 = {}
pred3 = {}
# baselines
mean_pred = np.full(len(y_test), y_train.mean())
persist = y_all.shift(1).reindex(X_test.index).fillna(y_train.mean()).values
res3["Mean baseline"] = metrics(y_test, mean_pred); pred3["Mean baseline"] = mean_pred
res3["Persistence (t-1)"] = metrics(y_test, persist); pred3["Persistence (t-1)"] = persist

fitted = {}
for name in ["RF", "HGB"]:
    p = make_pipeline(name, "median", SEED).fit(X_train, y_train)
    fitted[name] = p
    pred3[name] = p.predict(X_test)
    res3[name] = metrics(y_test, pred3[name])

tab3 = pd.DataFrame(res3).T.round(3)
tab3["RMSE/MAE"] = (tab3["RMSE"] / tab3["MAE"]).round(2)
print("Table 3 - Chronological hold-out, median imputation")
display(tab3)
for k, v in res3.items():
    log("Step 3 baseline/models", "chronological, median imputer", k, v["MAE"], v["RMSE"])

# light time-aware tuning check on the training block only (no test data involved)
tscv = TimeSeriesSplit(n_splits=4)
cv_rows = []
for name in ["RF", "HGB"]:
    for tr, va in tscv.split(X_train):
        p = make_pipeline(name, "median", SEED).fit(X_train.iloc[tr], y_train.iloc[tr])
        cv_rows.append(dict(model=name, RMSE=np.sqrt(mean_squared_error(y_train.iloc[va], p.predict(X_train.iloc[va])))))
cv = pd.DataFrame(cv_rows).groupby("model")["RMSE"].agg(["mean", "std"]).round(3)
print("\nTimeSeriesSplit(4) RMSE inside the training block (stability check, no test data):")
display(cv)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 4.2), gridspec_kw={"width_ratios": [1, 1.6]})
names = list(res3); x = np.arange(len(names)); w = 0.38
ax[0].bar(x - w/2, [res3[n]["MAE"] for n in names], w, label="MAE", color="#4C78A8")
ax[0].bar(x + w/2, [res3[n]["RMSE"] for n in names], w, label="RMSE", color="#F58518")
ax[0].set_xticks(x); ax[0].set_xticklabels(names, rotation=20, ha="right")
ax[0].set(title="Fig 2a - Supervised comparison (test block)", ylabel="mg/m3"); ax[0].legend()

last = X_test.index[-24 * 14:]
ax[1].plot(last, y_test.loc[last], color="black", lw=1.2, label="actual")
ax[1].plot(last, pd.Series(pred3["RF"], index=X_test.index).loc[last], color="#4C78A8", lw=1, label="RF")
ax[1].plot(last, pd.Series(pred3["HGB"], index=X_test.index).loc[last], color="#F58518", lw=1, label="HGB")
ax[1].set(title="Fig 2b - Predicted vs actual, last 14 days of the test block", ylabel="CO (mg/m3)"); ax[1].legend()
plt.tight_layout(); plt.savefig("figures/fig2_supervised_comparison.png", bbox_inches="tight"); plt.show()

## Step 4 - Unsupervised method: Isolation Forest

The detector is fitted **without the target** and on **training features only** (median imputation + standardisation, fitted on train). We use the physical measurements only (sensors, T, RH, AH), not the calendar columns, so that "anomalous" means "unusual physical state".

We first **characterise** the output (how many, when, which sensors). Only afterwards do we compare it with the supervised results.

In [ ]:
iso_prep = make_iso_prep().fit(X_train[SENSOR_COLS])
Z_train = iso_prep.transform(X_train[SENSOR_COLS])
Z_test = iso_prep.transform(X_test[SENSOR_COLS])

iso = IsolationForest(n_estimators=300, contamination=IF_CONTAM_MAIN, random_state=SEED, n_jobs=-1).fit(Z_train)
score_train = iso.decision_function(Z_train)          # lower = more anomalous
flag_train = iso.predict(Z_train) == -1
print(f"Flagged in training block: {flag_train.sum()} of {len(flag_train)} ({flag_train.mean():.1%})  | contamination = {IF_CONTAM_MAIN}")

# --- when do anomalies occur? ---
by_hour = pd.Series(flag_train, index=X_train.index).groupby(X_train.index.hour).mean()
by_month = pd.Series(flag_train, index=X_train.index).groupby(X_train.index.to_period("M")).mean()
print("\nHours with the highest anomaly rate:", by_hour.sort_values(ascending=False).head(4).round(3).to_dict())
print("Months with the highest anomaly rate:", {str(k): round(v, 3) for k, v in by_month.sort_values(ascending=False).head(3).items()})

# --- which sensors drive them? (mean |z| flagged minus unflagged) ---
zdf = pd.DataFrame(Z_train, columns=SENSOR_COLS, index=X_train.index)
drivers = (zdf[flag_train].abs().mean() - zdf[~flag_train].abs().mean()).sort_values(ascending=False).rename("excess |z| when flagged")
print("\nFeatures that drive the anomalies:")
display(drivers.round(2).to_frame())

# --- post-hoc: are anomalies faults or real episodes? (target used ONLY for this diagnosis) ---
y_flag, y_norm = y_train[flag_train], y_train[~flag_train]
q90 = y_train.quantile(0.90)
print(f"\nPost-hoc, CO of flagged hours:   median={y_flag.median():.2f}, share above the 90th percentile = {(y_flag > q90).mean():.1%}")
print(f"Post-hoc, CO of unflagged hours: median={y_norm.median():.2f}, share above the 90th percentile = {(y_norm > q90).mean():.1%}")

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(15, 9))
ax[0, 0].hist(score_train, bins=60, color="#4C78A8")
ax[0, 0].axvline(0, color="red", ls="--", label="flag threshold (score < 0)")
ax[0, 0].set(title="Fig 3a - Anomaly score distribution (train)", xlabel="decision_function (lower = more anomalous)"); ax[0, 0].legend()

pca = PCA(n_components=2, random_state=SEED).fit(Z_train)
P = pca.transform(Z_train)
ax[0, 1].scatter(P[~flag_train, 0], P[~flag_train, 1], s=5, c="#BBBBBB", label="normal")
ax[0, 1].scatter(P[flag_train, 0], P[flag_train, 1], s=10, c="#E45756", label="flagged")
ax[0, 1].set(title=f"Fig 3b - PCA view ({pca.explained_variance_ratio_.sum():.0%} of variance)", xlabel="PC1", ylabel="PC2"); ax[0, 1].legend()

ax[1, 0].plot(y_train.index, y_train, lw=0.4, color="#999999")
ax[1, 0].scatter(y_train.index[flag_train], y_train[flag_train], s=8, c="#E45756", label="flagged hours")
ax[1, 0].set(title=f"Fig 3c - Flagged hours over time ({TARGET} shown for context)", ylabel="mg/m3"); ax[1, 0].legend()

ax[1, 1].bar(by_hour.index, by_hour.values, color="#54A24B")
ax[1, 1].set(title="Fig 3d - Anomaly rate by hour of day", xlabel="hour", ylabel="share flagged")
plt.tight_layout(); plt.savefig("figures/fig3_isolation_forest.png", bbox_inches="tight"); plt.show()

In [ ]:
# ---- Only now: compare the unsupervised output with supervised results (on the test block) ----
Z_test_scores = iso.decision_function(Z_test)
flag_test = iso.predict(Z_test) == -1
resid = np.abs(y_test.values - pred3["RF"])
cmp = pd.DataFrame({"n": [flag_test.sum(), (~flag_test).sum()],
                    "mean |RF residual|": [resid[flag_test].mean(), resid[~flag_test].mean()],
                    "median |RF residual|": [np.median(resid[flag_test]) if flag_test.any() else np.nan, np.median(resid[~flag_test])]},
                   index=["flagged test hours", "unflagged test hours"]).round(3)
display(cmp)
rho, pval = spearmanr(Z_test_scores, resid)
print(f"Spearman correlation between anomaly score and |residual| on test: rho={rho:.3f} (p={pval:.2g})  "
      "-> negative rho means 'more anomalous = larger error'.")
IF_FACTS = dict(flag_train_rate=flag_train.mean(), share_flag_high=(y_flag > q90).mean(), share_norm_high=(y_norm > q90).mean(),
                resid_flag=resid[flag_test].mean() if flag_test.any() else np.nan, resid_norm=resid[~flag_test].mean(), rho=rho)

## Step 5 - The three compulsory experiments

Each experiment is repeated with the seeds in `SEEDS` and reported as *mean +/- std*, so we can tell a real effect from run-to-run noise. The chronological hold-out protocol and the untouched test block are the reference throughout.

In [ ]:
def summarize(rows, by):
    d = pd.DataFrame(rows)
    g = d.groupby(by).agg(MAE_mean=("MAE", "mean"), MAE_std=("MAE", "std"),
                          RMSE_mean=("RMSE", "mean"), RMSE_std=("RMSE", "std"), R2_mean=("R2", "mean"))
    return g.round(3)

def nearest_gap(train_t, test_t):
    tr = np.sort(train_t)
    pos = np.searchsorted(tr, test_t)
    left = tr[np.clip(pos - 1, 0, len(tr) - 1)]; right = tr[np.clip(pos, 0, len(tr) - 1)]
    return np.minimum(np.abs(test_t - left), np.abs(test_t - right))

### Experiment 1 - Validation protocol: chronological hold-out vs random split

Same models, same features, same train/test sizes, same median imputer. The only change is how the test set is chosen.

In [ ]:
rows1 = []
for seed in SEEDS:
    for m in ["RF", "HGB"]:
        p = make_pipeline(m, "median", seed).fit(X_train, y_train)
        rows1.append(dict(protocol="chronological", model=m, seed=seed, **metrics(y_test, p.predict(X_test))))
    Xr_tr, Xr_te, yr_tr, yr_te = train_test_split(X_v, y_v, test_size=1 - TRAIN_FRAC, shuffle=True, random_state=seed)
    for m in ["RF", "HGB"]:
        p = make_pipeline(m, "median", seed).fit(Xr_tr, yr_tr)
        rows1.append(dict(protocol="random", model=m, seed=seed, **metrics(yr_te, p.predict(Xr_te))))

res1 = summarize(rows1, ["model", "protocol"])
print("Table E1 - chronological vs random split (mean/std over seeds)")
display(res1)
for (m, prot), r in res1.iterrows():
    log("Exp 1 validation protocol", prot, m, r.MAE_mean, r.RMSE_mean, f"R2={r.R2_mean}")

# ---- evidence for WHY they differ ----
t_hours = (X_v.index - X_v.index[0]).total_seconds().values / 3600
gap_chrono = nearest_gap(t_hours[:n_train], t_hours[n_train:])
pos_tr = X_v.index.get_indexer(Xr_tr.index); pos_te = X_v.index.get_indexer(Xr_te.index)
gap_rand = nearest_gap(t_hours[pos_tr], t_hours[pos_te])
print(f"\nDistance (hours) from each test point to the nearest training point:")
print(f"  chronological: median={np.median(gap_chrono):.0f} h, mean={gap_chrono.mean():.0f} h, max={gap_chrono.max():.0f} h")
print(f"  random       : median={np.median(gap_rand):.0f} h, mean={gap_rand.mean():.1f} h, max={gap_rand.max():.0f} h")
print(f"Lag-1 autocorrelation of {TARGET}: {y_all.autocorr(1):.3f} -> a neighbouring hour is almost a copy of the test hour.")
print(f"Mean {TARGET}: train block={y_train.mean():.3f}, test block={y_test.mean():.3f} (seasonal drift)")
E1_FACTS = dict(gap_chrono=np.median(gap_chrono), gap_rand=np.median(gap_rand), ac1=y_all.autocorr(1))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.2))
x = np.arange(2); w = 0.38
for i, prot in enumerate(["chronological", "random"]):
    vals = [res1.loc[(m, prot), "RMSE_mean"] for m in ["RF", "HGB"]]
    errs = [res1.loc[(m, prot), "RMSE_std"] for m in ["RF", "HGB"]]
    ax[0].bar(x + (i - 0.5) * w, vals, w, yerr=errs, capsize=3, label=prot, color=["#4C78A8", "#F58518"][i])
ax[0].set_xticks(x); ax[0].set_xticklabels(["RF", "HGB"])
ax[0].set(title="Fig 4a - Exp 1: test RMSE by validation protocol", ylabel="RMSE (mg/m3)"); ax[0].legend()
ax[1].hist(gap_rand, bins=np.arange(0, 12) - 0.5, alpha=0.8, label="random split", color="#F58518")
ax[1].set(title="Fig 4b - Gap to nearest training hour (random split)", xlabel="hours", ylabel="test points")
ax[1].set_xlim(-1, 12); ax[1].legend()
plt.tight_layout(); plt.savefig("figures/fig4_exp1_validation.png", bbox_inches="tight"); plt.show()

### Experiment 2 - Missing-value strategy

Three defensible imputers are compared, plus HGB's native handling of `NaN`. **All are fitted on training data only.**

| Imputer | Idea | Leakage safeguard |
|---|---|---|
| `median` | training median of each column | statistics from train only (inside the pipeline) |
| `ffill` (causal) | carry the last known value forward (max `FFILL_LIMIT` hours) | only **past** values are used, so no future information reaches a test row; a train-median fallback fills what remains |
| `knn` | 5-nearest-neighbour imputation on standardised features | scaler and neighbours come from train only |
| `native NaN` (HGB only) | tree learns a default direction for missing values | no imputation at all |

We deliberately do **not** use linear interpolation: it needs the *next* known value, which would leak future data into the test set.

In [ ]:
X_ffill = X_all.copy()
X_ffill[SENSOR_COLS] = X_all[SENSOR_COLS].ffill(limit=FFILL_LIMIT)    # causal: past values only
Xf_v = X_ffill[valid]
Xf_train, Xf_test = Xf_v.iloc[:n_train], Xf_v.iloc[n_train:]

variants = [("median", "RF", X_train, X_test, "median"), ("median", "HGB", X_train, X_test, "median"),
            ("ffill (causal)", "RF", Xf_train, Xf_test, "median"), ("ffill (causal)", "HGB", Xf_train, Xf_test, "median"),
            ("knn", "RF", X_train, X_test, "knn"), ("knn", "HGB", X_train, X_test, "knn"),
            ("native NaN", "HGB", X_train, X_test, "none")]
rows2 = []
for seed in SEEDS:
    for label, m, Xtr, Xte, imp in variants:
        p = make_pipeline(m, imp, seed).fit(Xtr, y_train)
        rows2.append(dict(imputer=label, model=m, seed=seed, **metrics(y_test, p.predict(Xte))))

res2 = summarize(rows2, ["model", "imputer"])
print("Table E2 - imputation strategies (chronological hold-out, mean/std over seeds)")
display(res2)
for (m, imp), r in res2.iterrows():
    log("Exp 2 missing values", imp, m, r.MAE_mean, r.RMSE_mean, f"R2={r.R2_mean}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.2), sharey=True)
for a, m in zip(ax, ["RF", "HGB"]):
    sub = res2.loc[m]
    xs = np.arange(len(sub))
    a.bar(xs - 0.19, sub["MAE_mean"], 0.38, yerr=sub["MAE_std"], capsize=3, label="MAE", color="#4C78A8")
    a.bar(xs + 0.19, sub["RMSE_mean"], 0.38, yerr=sub["RMSE_std"], capsize=3, label="RMSE", color="#F58518")
    a.set_xticks(xs); a.set_xticklabels(sub.index, rotation=15)
    a.set(title=f"Fig 5 - Exp 2: {m}", ylabel="mg/m3"); a.legend()
plt.tight_layout(); plt.savefig("figures/fig5_exp2_imputation.png", bbox_inches="tight"); plt.show()

### Experiment 3 - Anomaly analysis (Isolation Forest contamination)

For `contamination` in {1%, 3%, 5%}: fit Isolation Forest on the training block, **remove the flagged points from the training set only**, retrain both models, and score them on the **complete, untouched test block** (headline metric). A 0% row (no removal) is the reference.

As a secondary diagnostic only, we also report the error on test hours that the same detector flags, to see whether removing training anomalies hurts exactly where anomalies occur. Test points are never removed.

In [ ]:
rows3 = []
for seed in SEEDS:
    pre = make_iso_prep().fit(X_train[SENSOR_COLS])
    Ztr, Zte = pre.transform(X_train[SENSOR_COLS]), pre.transform(X_test[SENSOR_COLS])
    for c in [0.0] + CONTAMINATIONS:
        if c == 0.0:
            keep = np.ones(len(X_train), dtype=bool); ftest = np.zeros(len(X_test), dtype=bool)
        else:
            iso_c = IsolationForest(n_estimators=300, contamination=c, random_state=seed, n_jobs=-1).fit(Ztr)
            keep = iso_c.predict(Ztr) == 1
            ftest = iso_c.predict(Zte) == -1
        for m in ["RF", "HGB"]:
            p = make_pipeline(m, "median", seed).fit(X_train[keep], y_train[keep])
            pr = p.predict(X_test)
            r = metrics(y_test, pr)                                  # headline: full untouched test block
            e = (y_test.values - pr) ** 2
            rows3.append(dict(contamination=c, model=m, seed=seed, n_removed=int((~keep).sum()), **r,
                              RMSE_test_flagged=np.sqrt(e[ftest].mean()) if ftest.any() else np.nan,
                              RMSE_test_unflagged=np.sqrt(e[~ftest].mean())))

d3 = pd.DataFrame(rows3)
res3e = d3.groupby(["model", "contamination"]).agg(n_removed=("n_removed", "mean"),
        MAE_mean=("MAE", "mean"), MAE_std=("MAE", "std"), RMSE_mean=("RMSE", "mean"), RMSE_std=("RMSE", "std"),
        R2_mean=("R2", "mean"), RMSE_test_flagged=("RMSE_test_flagged", "mean"), RMSE_test_unflagged=("RMSE_test_unflagged", "mean")).round(3)
print("Table E3 - training-set anomaly removal (headline = full untouched test block)")
display(res3e)
for (m, c), r in res3e.iterrows():
    log("Exp 3 anomaly removal", f"contamination={c:.0%}", m, r.MAE_mean, r.RMSE_mean, f"removed~{int(r.n_removed)} train rows")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.2), sharey=True)
for a, m in zip(ax, ["RF", "HGB"]):
    sub = res3e.loc[m]
    xs = np.arange(len(sub))
    a.bar(xs, sub["RMSE_mean"], yerr=sub["RMSE_std"], capsize=4, color="#4C78A8")
    a.set_xticks(xs); a.set_xticklabels([f"{c:.0%}" for c in sub.index])
    lo = (sub["RMSE_mean"] - sub["RMSE_std"]).min() * 0.9
    a.set_ylim(lo, (sub["RMSE_mean"] + sub["RMSE_std"]).max() * 1.05)
    a.set(title=f"Fig 6 - Exp 3: {m}, test RMSE vs contamination", xlabel="contamination (0% = no removal)", ylabel="RMSE (mg/m3)")
plt.tight_layout(); plt.savefig("figures/fig6_exp3_anomaly_removal.png", bbox_inches="tight"); plt.show()

## Step 6 - Scientific interpretation

The cell below computes the evidence from the results above and gives a verdict on H1 and H2. **Read the numbers, then write your own sentences in the markdown answers that follow** - unsupported claims receive no credit.

In [ ]:
def rel(a, b): return (a - b) / b

print("=== H1: random split gives lower error than chronological split ===")
h1_rows = []
for m in ["RF", "HGB"]:
    c = res1.loc[(m, "chronological")]; r = res1.loc[(m, "random")]
    h1_rows.append(r.RMSE_mean < c.RMSE_mean - max(c.RMSE_std, r.RMSE_std if not np.isnan(r.RMSE_std) else 0))
    print(f"{m}: RMSE chronological={c.RMSE_mean:.3f}  random={r.RMSE_mean:.3f}  ->  random is {abs(rel(r.RMSE_mean, c.RMSE_mean)):.0%} "
          f"{'lower' if r.RMSE_mean < c.RMSE_mean else 'HIGHER'}")
print("H1 verdict:", "SUPPORTED" if all(h1_rows) else ("PARTIALLY supported" if any(h1_rows) else "NOT supported"))
print(f"Evidence: median gap to nearest training hour = {E1_FACTS['gap_rand']:.0f} h (random) vs {E1_FACTS['gap_chrono']:.0f} h (chronological); "
      f"lag-1 autocorrelation = {E1_FACTS['ac1']:.2f}")

print("\n=== H2: removing flagged training anomalies does not noticeably reduce test error ===")
h2_rows = []
for m in ["RF", "HGB"]:
    sub = res3e.loc[m]
    base = sub.loc[0.0]
    best_c = sub.loc[CONTAMINATIONS, "RMSE_mean"].idxmin()
    best = sub.loc[best_c]
    gain = (base.RMSE_mean - best.RMSE_mean) / base.RMSE_mean
    noise = max(base.RMSE_std, best.RMSE_std)
    negligible = (gain < 0.03) or ((base.RMSE_mean - best.RMSE_mean) < noise)
    h2_rows.append(negligible)
    print(f"{m}: no removal RMSE={base.RMSE_mean:.3f}; best removal ({best_c:.0%}) RMSE={best.RMSE_mean:.3f}; "
          f"relative gain={gain:+.1%}; seed noise (std)={noise:.3f}")
print("H2 verdict:", "SUPPORTED" if all(h2_rows) else ("PARTIALLY supported" if any(h2_rows) else "NOT supported"))
print(f"\nAnomaly evidence (Step 4): flagged hours have {IF_FACTS['share_flag_high']:.0%} of CO above the 90th percentile "
      f"vs {IF_FACTS['share_norm_high']:.0%} for unflagged hours; mean |RF residual| flagged={IF_FACTS['resid_flag']:.3f} vs unflagged={IF_FACTS['resid_norm']:.3f}.")

### Written answers (complete after reading the numbers above)

**Q1 - Why might a random split be optimistic?**
*Points to develop with your numbers:* (i) hourly CO is highly autocorrelated, so each randomly chosen test hour has training hours immediately before and after it - the model can interpolate instead of forecast (see Fig 4b and the lag-1 autocorrelation); (ii) the real use case is predicting the future, but a random split also leaks seasonal/regime information because every period appears in training; (iii) the chronological test block may contain conditions (e.g. months, levels) absent from training - drift - which a random split hides. State the measured RMSE gap for RF and HGB.

**Q2 - Do anomalies always represent faulty sensors?**
*Points to develop:* No. Isolation Forest only says a reading is *statistically unusual in feature space*. Use your Step 4 evidence: at which hours/months do flags concentrate (Fig 3c-3d), which features drive them, and whether flagged hours carry high CO (post-hoc) - rush-hour peaks, cold and humid nights, and seasonal extremes are real events, not faults. Mention that removing them from training did or did not help (Exp 3).

**Q3 - What conclusions are justified without anomaly ground truth?**
*Points to develop:* Only descriptive ones: how many points are unusual, when and in which sensors, and that supervised error differs (or not) on them. We cannot compute precision/recall or claim a sensor failed, because no labels say which readings are faults; a flagged point might be a true pollution spike. Any conclusion about "faulty sensors" would need maintenance logs or a co-located reference analyser. Also note that contamination is a *user-chosen* fraction, not a measured fault rate.

### Limitations to mention in the report
* One city, 13 months, one target; results may not transfer to other pollutants or seasons.
* The test block (last 25%) covers only the winter-spring months, so seasonal drift and test-set size both affect the numbers.
* Hyper-parameters were fixed (not heavily tuned), and only three seeds were used to estimate noise.
* Causal forward-fill was limited to `FFILL_LIMIT` hours; longer gaps fall back to the training median.
* Isolation Forest used a single feature set and scaling choice.

## Step 7 - Report scaffold, contributions and oral defence

### Experiment log

In [ ]:
log_df = pd.DataFrame(LOG)
log_df.to_csv("experiment_log.csv", index=False)
display(log_df)
print("Saved experiment_log.csv and the figures/ folder (fig1 ... fig6).")

# Part 2 – MLflow tracking + online REST API

Everything below runs **in the same runtime** as Steps 0–7 above (it reuses `X_train`, `make_pipeline`, `metrics`, `rows1–3`, `res1`, `res2`, `res3e`, `figures/`, `experiment_log.csv`). Use **Runtime → Run all**.

| Step | What it does |
|---|---|
| 8 | Install MLflow, FastAPI, uvicorn |
| 9 | Log the three experiments to MLflow (one parent run per experiment, nested runs per setting/model/seed) |
| 10 | Train, log and **register** the final model in the MLflow Model Registry |
| 11 | Build the FastAPI service (model loaded back from the registry) |
| 12 | Start the API inside Colab and test it locally |
| 13 | Open the interactive API page (Swagger) inside Colab |
| 14 | Call the API from the notebook and compare predictions with true values |
| 15 | Streamlit web app that calls the API |
| 16–17 | (Optional) MLflow UI · save everything to Google Drive |

Everything runs inside Colab – no ngrok, no Cloudflare, no account needed.

## Step 8 – Install the MLOps packages

In [ ]:
!pip -q install "mlflow>=3" fastapi uvicorn httpx

## Step 9 – Track the three experiments with MLflow
Tracking store: a local SQLite file `mlflow.db` (needed for the Model Registry).

In [ ]:
import os, math, warnings
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
import mlflow, mlflow.sklearn
from mlflow.models import infer_signature
warnings.filterwarnings("ignore", module="mlflow")

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("Group4_AirQuality_CO")

CONFIG = dict(seed=SEED, seeds=str(SEEDS), target=TARGET, train_frac=TRAIN_FRAC,
              ffill_limit=FFILL_LIMIT, contaminations=str(CONTAMINATIONS),
              excluded_proxy=str(PROXY_COLS), n_train=len(X_train), n_test=len(X_test))
METRIC_KEYS = ["MAE", "RMSE", "R2", "MedAE", "RMSE_test_flagged", "RMSE_test_unflagged", "n_removed"]

def _num(v):
    try:
        v = float(v)
        return None if math.isnan(v) else v
    except (TypeError, ValueError):
        return None

def log_experiment(name, rows, param_keys, summary=None):
    with mlflow.start_run(run_name=name):
        mlflow.log_params(CONFIG)
        mlflow.set_tag("experiment_block", name)
        if summary is not None:
            summary.to_csv(f"{name}_summary.csv")
            mlflow.log_artifact(f"{name}_summary.csv")
        for r in rows:
            with mlflow.start_run(run_name=" | ".join(f"{k}={r[k]}" for k in param_keys), nested=True):
                mlflow.log_params({k: r[k] for k in param_keys})
                mlflow.log_metrics({k: _num(r[k]) for k in METRIC_KEYS if k in r and _num(r[k]) is not None})

log_experiment("Exp1_validation_protocol", rows1, ["protocol", "model", "seed"], res1)
log_experiment("Exp2_missing_values", rows2, ["imputer", "model", "seed"], res2)
log_experiment("Exp3_anomaly_removal", rows3, ["contamination", "model", "seed"], res3e)
print("Experiments logged to MLflow.")

## Step 10 – Train, log and register the final model
RF + KNN imputer gave the best chronological RMSE in Experiment 2 — change `FINAL_MODEL` / `FINAL_IMPUTER` if your results differ.

MLflow 3 saves scikit-learn models with **skops** (a safe format that refuses unknown object types). Tree models contain internal tree objects, so we explicitly list them as trusted — safe here because we created the model ourselves.

In [ ]:
FINAL_MODEL, FINAL_IMPUTER = "RF", "knn"
MODEL_NAME = "group4_co_gt_predictor"
FEATURES = list(X_train.columns)

# internal sklearn types used by RandomForest / HistGradientBoosting (we built the model, so we trust them)
SKOPS_TRUSTED = [
    "sklearn.tree._tree.Tree",
    "sklearn.ensemble._hist_gradient_boosting.predictor.TreePredictor",
    "sklearn.ensemble._hist_gradient_boosting.binning._BinMapper",
    "sklearn._loss.link.IdentityLink", "sklearn._loss.link.Interval",
    "sklearn._loss.loss.HalfSquaredError", "sklearn._loss._loss.CyHalfSquaredError",
    "numpy.dtype",
]

with mlflow.start_run(run_name=f"final_{FINAL_MODEL}_{FINAL_IMPUTER}") as run:
    pipe = make_pipeline(FINAL_MODEL, FINAL_IMPUTER, SEED).fit(X_train, y_train)
    pred = pipe.predict(X_test)
    final_metrics = metrics(y_test, pred)
    mlflow.log_params({**CONFIG, "model": FINAL_MODEL, "imputer": FINAL_IMPUTER, "features": ",".join(FEATURES)})
    mlflow.log_metrics({k: float(v) for k, v in final_metrics.items()})
    if os.path.isdir("figures"):
        mlflow.log_artifacts("figures", artifact_path="figures")
    if os.path.exists("experiment_log.csv"):
        mlflow.log_artifact("experiment_log.csv")
    mlflow.sklearn.log_model(pipe, name="model",
                             signature=infer_signature(X_test, pred),
                             input_example=X_test.dropna().head(3),
                             registered_model_name=MODEL_NAME,
                             skops_trusted_types=SKOPS_TRUSTED)
    RUN_ID = run.info.run_id

print("Run id:", RUN_ID)
print({k: round(float(v), 3) for k, v in final_metrics.items()})

## Step 11 – Build the FastAPI service
Inputs are only the low-cost sensors + T, RH, AH (no `PT08.S1(CO)` or reference analyzers → no leakage). Hour / day-of-week / month come from the timestamp. Missing sensor values are allowed — the pipeline's imputer fills them.

In [ ]:
from datetime import datetime
from typing import List, Optional
import numpy as np, pandas as pd
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

try:
    served_model = mlflow.sklearn.load_model(f"models:/{MODEL_NAME}/latest")
    MODEL_SOURCE = f"MLflow registry: models:/{MODEL_NAME}/latest"
except Exception as e:                      # safety net: serve the model trained above
    print("Registry load failed, serving in-memory model:", str(e)[:200])
    served_model, MODEL_SOURCE = pipe, f"in-memory pipeline (run {RUN_ID})"
print("Serving:", MODEL_SOURCE)

# JSON field name -> training column name
COL_MAP = {"PT08_S2_NMHC": "PT08.S2(NMHC)", "PT08_S3_NOx": "PT08.S3(NOx)",
           "PT08_S4_NO2": "PT08.S4(NO2)", "PT08_S5_O3": "PT08.S5(O3)",
           "T": "T", "RH": "RH", "AH": "AH"}
missing = [c for c in FEATURES if c not in list(COL_MAP.values()) + ["hour", "dayofweek", "month"]]
assert not missing, f"Update COL_MAP: features not provided by the API -> {missing}"

class Reading(BaseModel):
    timestamp: datetime
    PT08_S2_NMHC: Optional[float] = None
    PT08_S3_NOx: Optional[float] = None
    PT08_S4_NO2: Optional[float] = None
    PT08_S5_O3: Optional[float] = None
    T: Optional[float] = None
    RH: Optional[float] = None
    AH: Optional[float] = None
    model_config = {"json_schema_extra": {"example": {
        "timestamp": "2005-03-01T08:00:00", "PT08_S2_NMHC": 1046.0, "PT08_S3_NOx": 1056.0,
        "PT08_S4_NO2": 1692.0, "PT08_S5_O3": 1268.0, "T": 13.6, "RH": 48.9, "AH": 0.7578}}}

def to_frame(readings):
    rows = []
    for r in readings:
        d = {col: (np.nan if getattr(r, f) is None else getattr(r, f)) for f, col in COL_MAP.items()}
        d.update(hour=r.timestamp.hour, dayofweek=r.timestamp.weekday(), month=r.timestamp.month)
        rows.append(d)
    X = pd.DataFrame(rows)[FEATURES]
    sensors = [c for c in COL_MAP.values() if c in FEATURES]
    if X[sensors].isna().all(axis=1).any():
        raise HTTPException(status_code=422, detail="Each reading needs at least one sensor value.")
    return X

app = FastAPI(title="Group 4 – CO(GT) predictor", version="1.0",
              description="Predicts hourly CO(GT) in mg/m³ from the low-cost sensor array (UCI Air Quality). "
                          "Model tracked and registered with MLflow.")

@app.get("/")
def root():
    return {"message": "Group 4 CO(GT) API – open /docs to try it", "endpoints": ["/health", "/predict", "/predict_batch"]}

@app.get("/health")
def health():
    return {"status": "ok", "model": MODEL_NAME, "source": MODEL_SOURCE, "run_id": RUN_ID,
            "test_metrics": {k: round(float(v), 3) for k, v in final_metrics.items()}, "features": FEATURES}

@app.post("/predict")
def predict(reading: Reading):
    y = served_model.predict(to_frame([reading]))[0]
    return {"timestamp": reading.timestamp, "CO_GT_pred_mg_m3": round(float(y), 3)}

@app.post("/predict_batch")
def predict_batch(readings: List[Reading]):
    ys = served_model.predict(to_frame(readings))
    return [{"timestamp": r.timestamp, "CO_GT_pred_mg_m3": round(float(y), 3)} for r, y in zip(readings, ys)]

## Step 12 – Start the API inside Colab
Safe to re-run: if the server is already up it is not started twice.

In [ ]:
import threading, time, socket, subprocess, requests, uvicorn

def port_open(port):
    with socket.socket() as s:
        return s.connect_ex(("127.0.0.1", port)) == 0

if not port_open(8000):
    server = uvicorn.Server(uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="warning"))
    threading.Thread(target=server.run, daemon=True).start()
    for _ in range(30):
        if port_open(8000): break
        time.sleep(0.5)
else:
    print("API already running on port 8000 (restart the runtime to load a new model).")

print(requests.get("http://127.0.0.1:8000/health").json())

## Step 13 – Open the API inside Colab
The API runs on the Colab machine (not on your computer). Colab's own proxy shows the interactive Swagger page **directly in the output below** – no ngrok, no Cloudflare. Use *Try it out* → *Execute* on `/predict`.

In [ ]:
from google.colab import output
output.serve_kernel_port_as_iframe(8000, path="/docs", height=900)

In [ ]:
# Same page in a separate browser tab (link works for you while this runtime is running)
output.serve_kernel_port_as_window(8000, path="/docs")

## Step 14 – Call the API from the notebook
`TestClient` sends real HTTP requests to the FastAPI app inside the notebook process (same validation, same responses as a deployed API).

In [ ]:
from fastapi.testclient import TestClient
client = TestClient(app)

print("Health:", client.get("/health").json()["status"])

sample = {"timestamp": "2005-03-01T08:00:00", "PT08_S2_NMHC": 1046.0, "PT08_S3_NOx": 1056.0,
          "PT08_S4_NO2": 1692.0, "PT08_S5_O3": 1268.0, "T": 13.6, "RH": 48.9, "AH": 0.7578}
print("Single prediction:", client.post("/predict", json=sample).json())

# 10 real test hours as one batch, compared with the true CO(GT)
batch = []
for ts, row in X_test.head(10).iterrows():
    item = {"timestamp": pd.Timestamp(ts).isoformat()}
    for f, col in COL_MAP.items():
        item[f] = None if pd.isna(row[col]) else float(row[col])
    batch.append(item)
resp = client.post("/predict_batch", json=batch).json()
check = pd.DataFrame({"timestamp": [r["timestamp"] for r in resp],
                      "pred_api": [r["CO_GT_pred_mg_m3"] for r in resp],
                      "true_CO_GT": y_test.head(10).values})
check["abs_error"] = (check.pred_api - check.true_CO_GT).abs().round(3)
check["matches_notebook"] = np.isclose(check.pred_api, pipe.predict(X_test.head(10)).round(3), atol=1e-3)
display(check)

# missing sensor -> imputed; no sensors at all -> 422 error
print("Missing O3 sensor:", client.post("/predict", json={**sample, "PT08_S5_O3": None}).json())
r = client.post("/predict", json={"timestamp": "2005-03-01T08:00:00"})
print("No sensors:", r.status_code, r.json())

## Step 15 – Streamlit web app (runs inside Colab)
A user-friendly front-end: sliders for each sensor → the app calls the FastAPI `/predict` endpoint → shows the prediction. Also batch CSV predictions, predicted-vs-true chart and the MLflow runs.

First we save the info the app needs (input ranges, a test sample, the MLflow run table), then write `app.py`.

In [ ]:
import json
LABELS = {"PT08_S2_NMHC": "PT08.S2 (NMHC sensor)", "PT08_S3_NOx": "PT08.S3 (NOx sensor)",
          "PT08_S4_NO2": "PT08.S4 (NO2 sensor)", "PT08_S5_O3": "PT08.S5 (O3 sensor)",
          "T": "Temperature T (°C)", "RH": "Relative humidity RH (%)", "AH": "Absolute humidity AH"}
fields = {f: {"col": c, "label": LABELS[f],
              "min": float(X_train[c].quantile(0.01)), "max": float(X_train[c].quantile(0.99)),
              "median": float(X_train[c].median())} for f, c in COL_MAP.items()}
json.dump({"fields": fields, "default_ts": str(pd.Timestamp(X_test.index[0])),
           "target_median": float(y_train.median()),
           "model_label": f"{FINAL_MODEL} + {FINAL_IMPUTER} imputer"},
          open("app_meta.json", "w"), indent=1)

sample = X_test.head(24 * 14).copy()                    # two weeks of real test hours
out = pd.DataFrame({"timestamp": [pd.Timestamp(t).isoformat() for t in sample.index]})
for f, c in COL_MAP.items():
    out[f] = sample[c].values
out["true_CO_GT"] = y_test.loc[sample.index].values
out.to_csv("api_test_sample.csv", index=False)

runs = mlflow.search_runs(experiment_names=["Group4_AirQuality_CO"])
keep = [c for c in ["tags.mlflow.runName", "metrics.RMSE", "metrics.MAE", "metrics.R2", "start_time"] if c in runs]
runs[keep].rename(columns=lambda c: c.split(".")[-1]).to_csv("mlflow_runs.csv", index=False)
print("Saved app_meta.json, api_test_sample.csv, mlflow_runs.csv")

In [ ]:
%%writefile app.py
import json, os
import numpy as np, pandas as pd, requests, streamlit as st

API = "http://127.0.0.1:8000"          # the FastAPI service running in this Colab runtime
st.set_page_config(page_title="Group 4 – CO(GT) predictor", page_icon="🌫️", layout="wide")

@st.cache_data
def load_meta():
    with open("app_meta.json") as f:
        return json.load(f)

meta = load_meta()
FIELDS = meta["fields"]                 # JSON field -> {"col", "label", "min", "max", "median"}

def api_ok():
    try:
        return requests.get(f"{API}/health", timeout=3).json()
    except Exception:
        return None

st.title("🌫️ Group 4 – Air quality: CO(GT) predictor")
st.caption("UCI Air Quality · model tracked & registered with MLflow · served by FastAPI · UI in Streamlit")

health = api_ok()
if health is None:
    st.error("The API is not reachable. Run Step 12 (start the API) in the notebook, then refresh this page.")
    st.stop()

c1, c2, c3, c4 = st.columns(4)
m = health["test_metrics"]
c1.metric("Test MAE (mg/m³)", f'{m["MAE"]:.3f}')
c2.metric("Test RMSE (mg/m³)", f'{m["RMSE"]:.3f}')
c3.metric("Test R²", f'{m["R2"]:.3f}')
c4.metric("Model", meta["model_label"])

tab1, tab2, tab3 = st.tabs(["Single reading", "Batch (CSV)", "Test set & MLflow"])

with tab1:
    st.subheader("Predict CO(GT) for one hourly reading")
    left, right = st.columns([2, 1])
    with left:
        d = st.date_input("Date", value=pd.Timestamp(meta["default_ts"]).date())
        t = st.time_input("Hour", value=pd.Timestamp(meta["default_ts"]).time(), step=3600)
        values = {}
        cols = st.columns(2)
        for i, (field, info) in enumerate(FIELDS.items()):
            with cols[i % 2]:
                missing = st.checkbox(f"{info['label']} missing", key=f"miss_{field}")
                values[field] = None if missing else st.slider(
                    info["label"], float(info["min"]), float(info["max"]), float(info["median"]), key=field)
    with right:
        st.write("")
        if st.button("Predict", type="primary", width="stretch"):
            payload = {"timestamp": pd.Timestamp.combine(d, t).isoformat(), **values}
            r = requests.post(f"{API}/predict", json=payload, timeout=30)
            if r.ok:
                y = r.json()["CO_GT_pred_mg_m3"]
                st.metric("Predicted CO(GT)", f"{y:.2f} mg/m³")
                level = "low" if y < 2 else "moderate" if y < 4 else "high"
                st.write(f"Level: **{level}** (training median: {meta['target_median']:.2f} mg/m³)")
            else:
                st.error(r.json().get("detail", r.text))
            with st.expander("Request sent to the API"):
                st.json(payload)
        st.info("Tick “missing” to simulate a failed sensor – the model's imputer fills it.")

with tab2:
    st.subheader("Predict many readings from a CSV")
    st.write("Columns: `timestamp` plus any of " + ", ".join(f"`{f}`" for f in FIELDS))
    template = pd.read_csv("api_test_sample.csv").drop(columns=["true_CO_GT"]).head(24)
    st.download_button("Download example CSV (24 test hours)", template.to_csv(index=False),
                       "example_readings.csv", "text/csv")
    up = st.file_uploader("Upload CSV", type="csv")
    df = pd.read_csv(up) if up is not None else None
    if df is None and st.checkbox("Use the example file"):
        df = template
    if df is not None:
        rows = [{k: (None if pd.isna(v) else v) for k, v in r.items() if k in FIELDS or k == "timestamp"}
                for r in df.to_dict("records")]
        r = requests.post(f"{API}/predict_batch", json=rows, timeout=60)
        if r.ok:
            out = df.copy()
            out["CO_GT_pred_mg_m3"] = [p["CO_GT_pred_mg_m3"] for p in r.json()]
            st.line_chart(out.set_index(pd.to_datetime(out["timestamp"]))["CO_GT_pred_mg_m3"])
            st.dataframe(out, width="stretch")
            st.download_button("Download predictions", out.to_csv(index=False), "predictions.csv", "text/csv")
        else:
            st.error(r.text)

with tab3:
    st.subheader("API predictions vs true CO(GT) on the chronological test set")
    test = pd.read_csv("api_test_sample.csv")
    n = st.slider("Hours to show", 24, len(test), min(168, len(test)), step=24)
    sub = test.head(n)
    payload = [{k: (None if pd.isna(v) else v) for k, v in r.items() if k != "true_CO_GT"}
               for r in sub.to_dict("records")]
    preds = [p["CO_GT_pred_mg_m3"] for p in requests.post(f"{API}/predict_batch", json=payload, timeout=60).json()]
    chart = pd.DataFrame({"true CO(GT)": sub["true_CO_GT"].values, "predicted": preds},
                         index=pd.to_datetime(sub["timestamp"]))
    st.line_chart(chart)
    err = np.abs(chart["predicted"] - chart["true CO(GT)"])
    st.write(f"MAE on these {n} hours: **{err.mean():.3f} mg/m³**")

    st.subheader("MLflow runs")
    if os.path.exists("mlflow_runs.csv"):
        st.dataframe(pd.read_csv("mlflow_runs.csv"), width="stretch")
    st.caption(f"Registered model: `{health['model']}` · run id `{health['run_id']}` · source: {health['source']}")

In [ ]:
!pip -q install streamlit
import requests
if not port_open(8501):
    st_log = open("streamlit.log", "w")
    subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true",
                      "--server.enableCORS", "false", "--server.enableXsrfProtection", "false",
                      "--server.enableWebsocketCompression", "false",
                      "--browser.gatherUsageStats", "false"],
                     stdout=st_log, stderr=subprocess.STDOUT)
    for _ in range(60):
        if port_open(8501): break
        time.sleep(0.5)
try:
    print("Streamlit health:", requests.get("http://127.0.0.1:8501/_stcore/health", timeout=5).text)
except Exception as e:
    print("Streamlit not answering – log:\n", open("streamlit.log").read()[-2000:])
output.serve_kernel_port_as_iframe(8501, height=1000)   # app shown right here

In [ ]:
# Open the Streamlit app in its own browser tab (works while the runtime is running)
output.serve_kernel_port_as_window(8501)

## Step 16 (optional) – MLflow UI
Shows the MLflow dashboard (experiments, runs, metrics, registered model) inside Colab.

In [ ]:
if not port_open(5000):
    subprocess.Popen(["mlflow", "ui", "--backend-store-uri", "sqlite:///mlflow.db",
                      "--host", "0.0.0.0", "--port", "5000"],
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(40):
        if port_open(5000): break
        time.sleep(0.5)
output.serve_kernel_port_as_iframe(5000, height=800)

## Step 17 (optional) – Save MLflow runs and the model to Google Drive
Colab deletes local files when the runtime stops.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/group4_mlflow
!cp -r mlflow.db mlruns /content/drive/MyDrive/group4_mlflow/ 2>/dev/null; ls /content/drive/MyDrive/group4_mlflow